# Who will win? Baseline Modeling

## Loading in the data from local PostgreSQL database

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [2]:
query = """
SELECT *
FROM vw_game_model_features_v2
ORDER BY season, week;
"""

df = pd.read_sql(
    query,
    engine
)

df.shape

(26455, 47)

## Let's take a look at the data

In [4]:
print(df.shape,df.columns,sep="\n")

(26455, 47)
Index(['game_id', 'season', 'week', 'start_date', 'home_team_id', 'home_team',
       'away_team_id', 'away_team', 'home_conference', 'away_conference',
       'conference_game', 'neutral_site', 'home_pregame_elo',
       'away_pregame_elo', 'home_off_yards_per_play', 'home_off_ppa',
       'home_off_success_rate', 'home_off_explosive_rate',
       'home_off_early_down_ypp', 'home_off_third_down_rate',
       'home_def_yards_per_play_allowed', 'home_def_ppa_allowed',
       'home_def_success_rate_allowed', 'home_def_explosive_rate_allowed',
       'home_def_early_down_ypp_allowed', 'home_def_third_down_rate_allowed',
       'away_off_yards_per_play', 'away_off_ppa', 'away_off_success_rate',
       'away_off_explosive_rate', 'away_off_early_down_ypp',
       'away_off_third_down_rate', 'away_def_yards_per_play_allowed',
       'away_def_ppa_allowed', 'away_def_success_rate_allowed',
       'away_def_explosive_rate_allowed', 'away_def_early_down_ypp_allowed',
       'away_def

In [ ]:
df.info() #At the point of creating this notebook, I've only loaded play data for 2021-2025. Hence the massive amount of missing values

<class 'pandas.DataFrame'>
RangeIndex: 26455 entries, 0 to 26454
Data columns (total 47 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   game_id                           26455 non-null  int64  
 1   season                            26455 non-null  int64  
 2   week                              26455 non-null  int64  
 3   start_date                        26455 non-null  object 
 4   home_team_id                      26455 non-null  int64  
 5   home_team                         26455 non-null  str    
 6   away_team_id                      26455 non-null  int64  
 7   away_team                         26455 non-null  str    
 8   home_conference                   26376 non-null  str    
 9   away_conference                   26124 non-null  str    
 10  conference_game                   26455 non-null  bool   
 11  neutral_site                      26455 non-null  bool   
 12  home_pregame_el

In [7]:
df.head()

,game_id,season,week,start_date,home_team_id,home_team,away_team_id,away_team,home_conference,away_conference,...,away_def_third_down_rate_allowed,offensive_ppa_diff,defensive_ppa_allowed_diff,offensive_success_rate_diff,offensive_ypp_diff,pregame_elo_diff,home_points,away_points,home_point_margin,home_win
0,400795505,2015,1,2015-09-05 19:00:00-04:00,2617,Stephen F. Austin,2464,Northern Arizona,Southland,Big Sky,...,NaN,NaN,NaN,NaN,NaN,NaN,28.0,34.0,-6.0,0
1,400763497,2015,1,2015-09-05 15:30:00-04:00,218,Temple,213,Penn State,American Athletic,Big Ten,...,NaN,NaN,NaN,NaN,NaN,-67.0,27.0,10.0,17.0,1
2,400763406,2015,1,2015-09-05 15:30:00-04:00,2641,Texas Tech,2534,Sam Houston,Big 12,Southland,...,NaN,NaN,NaN,NaN,NaN,NaN,59.0,45.0,14.0,1
3,400756901,2015,1,2015-09-05 18:00:00-04:00,152,NC State,2653,Troy,ACC,Sun Belt,...,NaN,NaN,NaN,NaN,NaN,333.0,49.0,21.0,28.0,1
4,400764855,2015,1,2015-09-04 19:00:00-04:00,2567,SMU,239,Baylor,American Athletic,Big 12,...,NaN,NaN,NaN,NaN,NaN,-727.0,21.0,56.0,-35.0,0


In [8]:
df.season.value_counts().sort_index()

season
2015    1538
2016    1549
2017    1551
2018    1556
2019    1623
2020    1125
2021    2454
2022    3705
2023    3724
2024    3799
2025    3831
Name: count, dtype: int64

In [11]:
df.isna().mean().sort_values(ascending=False)

offensive_ypp_diff                  0.772822
offensive_success_rate_diff         0.772822
defensive_ppa_allowed_diff          0.772822
offensive_ppa_diff                  0.772822
away_def_early_down_ypp_allowed     0.771234
away_def_explosive_rate_allowed     0.771234
away_off_yards_per_play             0.771234
away_off_early_down_ypp             0.771234
away_off_explosive_rate             0.771234
away_off_success_rate               0.771234
away_off_ppa                        0.771234
away_def_success_rate_allowed       0.771234
away_off_third_down_rate            0.771234
away_def_ppa_allowed                0.771234
away_def_yards_per_play_allowed     0.771234
away_def_third_down_rate_allowed    0.771234
home_def_ppa_allowed                0.763599
home_def_third_down_rate_allowed    0.763599
home_def_yards_per_play_allowed     0.763561
home_def_success_rate_allowed       0.763561
home_def_early_down_ypp_allowed     0.763561
home_def_explosive_rate_allowed     0.763561
home_off_t

In [13]:
df.groupby("week")["home_off_ppa"].apply(lambda x: x.notna().mean())

week
1     0.020181
2     0.265774
3     0.255382
4     0.250000
5     0.244233
6     0.226753
7     0.230503
8     0.230216
9     0.229594
10    0.237983
11    0.252874
12    0.309524
13    0.433431
14    0.385686
15    0.263804
16    0.243902
20    0.000000
Name: home_off_ppa, dtype: float64

### The missing data from earlier seasons will be dealt with before modeling. Another thing to note is that week one will always have missing values because there is nothing to go off of yet.

## Time to Model

### Starting with a simple binary model. Does the home team win? Target: home_win

In [14]:
df.home_win.value_counts(normalize=True)

home_win
1    0.58144
0    0.41856
Name: proportion, dtype: float64

The home team wins 58% of the time in our dataset so that's the first baseline to beat. Can we do better than just predicting the home team wins every game?

Gonna start super simple and only use a few features to see what we can do.

In [15]:
model_df = df.dropna(subset=["home_off_ppa","away_off_ppa","home_def_ppa_allowed",
                             "away_def_ppa_allowed","pregame_elo_diff"]).copy()

model_df.shape

(3492, 47)

In [18]:
features = [
    "home_off_ppa",
    "away_off_ppa",
    "home_def_ppa_allowed",
    "away_def_ppa_allowed",
    "pregame_elo_diff"
]

X = model_df[features]
y = model_df["home_win"]

Gonna make 2021-2024 the training data and 2025 the test data to prevent leakage that would arise from random splitting.

In [19]:
train_time = model_df['season'] < 2025
valid_time = model_df['season'] == 2025

X_train = model_df.loc[train_time,features]
y_train = model_df.loc[train_time, 'home_win']

X_valid = model_df.loc[valid_time,features] 
y_valid = model_df.loc[valid_time, 'home_win']

In [23]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train,y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [24]:
preds = model.predict(X_valid)

In [25]:
pred_prob = model.predict_proba(X_valid)[:, 1]

In [26]:
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score
)

print(
    "Accuracy:",
    accuracy_score(y_valid, preds)
)

print(
    "ROC AUC:",
    roc_auc_score(
        y_valid,
        pred_prob
    )
)

print(
    "Log Loss:",
    log_loss(
        y_valid,
        pred_prob
    )
)

Accuracy: 0.7254901960784313
ROC AUC: 0.7983156601577653
Log Loss: 0.5335370563392615
